In [3]:
import os
from Crypto.Cipher import ChaCha20

In [4]:
# --- Stream cipher helpers ---
def encrypt_message(key, plaintext, nonce=None):
    """
    Encrypt a message with ChaCha20.
    Returns (ciphertext, nonce).
    If no nonce is given, a random 8-byte nonce is generated.
    """
    if nonce is None:
        nonce = os.urandom(8)
    cipher = ChaCha20.new(key=key, nonce=nonce)
    ciphertext = cipher.encrypt(plaintext)
    return ciphertext, nonce

def decrypt_message(key, ciphertext, nonce):
    """
    Decrypt a message with ChaCha20 given key and nonce.
    """
    cipher = ChaCha20.new(key=key, nonce=nonce)
    return cipher.decrypt(ciphertext)


# --- Simulated chat ---
print("=== Alice ↔ Bob Secure Chat Simulation ===")

# Shared symmetric key (e.g. derived via Diffie-Hellman beforehand)
shared_key = os.urandom(32)

# Conversation (sequence of messages)
conversation = [
    ("Alice", b"Hi Bob, how are you today?"),
    ("Bob",   b"Hey Alice, I am fine, thanks! What about you?"),
    ("Alice", b"I'm great. Want to meet later?"),
    ("Bob",   b"Sure, let's meet at 6 PM."),
]

# --- Messaging loop ---
transcript = []

for sender, msg in conversation:
    # Encrypt with a fresh nonce
    ciphertext, nonce = encrypt_message(shared_key, msg)
    
    # Store what was "sent" (header + ciphertext)
    transcript.append((sender, msg, ciphertext, nonce))
    
    # Show what goes over the network
    print(f"\n[{sender}] sends:")
    print("  Nonce (header):", nonce.hex())
    print("  Ciphertext:    ", ciphertext.hex())
    
    # Receiver decrypts
    decrypted = decrypt_message(shared_key, ciphertext, nonce)
    print("  Receiver decrypts ->", decrypted.decode())




=== Alice ↔ Bob Secure Chat Simulation ===

[Alice] sends:
  Nonce (header): 370a3c7fdc8aa331
  Ciphertext:     7dea22ae8d1677e1a9680e624b3fc3805353dae89b0ab6328e5f
  Receiver decrypts -> Hi Bob, how are you today?

[Bob] sends:
  Nonce (header): 554eba480f7dc90c
  Ciphertext:     6f8690d4de8dee343d4659b4ddecf6bacf3facb79d931038b2847e83c07736da1c4f29279a740a236dc95e218b
  Receiver decrypts -> Hey Alice, I am fine, thanks! What about you?

[Alice] sends:
  Nonce (header): a4fd2e7f88f17480
  Ciphertext:     9a2f80ca0e806c7fcb91c12f457ac626503b17f58be6ac68bb1c71fa18ac
  Receiver decrypts -> I'm great. Want to meet later?

[Bob] sends:
  Nonce (header): 2869612cf1068e8f
  Ciphertext:     73d8b4f1e87ed0be6f6a5c70ed7b48906601068c7a7f8f4149
  Receiver decrypts -> Sure, let's meet at 6 PM.


In [5]:
# --- Security failure: nonce reuse demo ---
print("\n=== Nonce reuse attack (two-time pad) ===")

m1 = b"Hello, this is Alice"
m2 = b"I need help from you"

# Bad practice: same nonce used for both
bad_nonce = os.urandom(8)

c1, _ = encrypt_message(shared_key, m1, nonce=bad_nonce)
c2, _ = encrypt_message(shared_key, m2, nonce=bad_nonce)

# Attacker computes XOR of ciphertexts
xor_ct = bytes([a ^ b for a, b in zip(c1, c2)])
xor_pt = bytes([a ^ b for a, b in zip(m1, m2)])

print("C1 XOR C2:", xor_ct)
print("M1 XOR M2:", xor_pt)

print("\n>>> Notice: C1 XOR C2 = M1 XOR M2. Reusing the same nonce leaks information!")


=== Nonce reuse attack (two-time pad) ===
C1 XOR C2: b'\x01E\x02\t\nH\x00\x1c\r\x05\x03\x00\x0f\x01O,L\x10\x0c\x10'
M1 XOR M2: b'\x01E\x02\t\nH\x00\x1c\r\x05\x03\x00\x0f\x01O,L\x10\x0c\x10'

>>> Notice: C1 XOR C2 = M1 XOR M2. Reusing the same nonce leaks information!
